# parameter-subclass-of-tensor — faded example 2: complete the trainable-params filter

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-subclass-of-tensor`. Running the beacon reports progress on the `Backprop: Parameter subclasses Tensor` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter subclasses Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-subclass-of-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-subclass-of-tensor"
DD_SUBTOPIC = "Backprop: Parameter subclasses Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Trainable params are the attributes that are strictly `isinstance(_, Parameter)`. Plain MiniTensor buffers and non-tensors must be skipped, which is why the filter checks the Parameter type, not MiniTensor.

## Faded exercise 2

Complete `trainable_params` so it yields only attributes that are Parameters. Fill in the type-check condition.

**Fill in:** the isinstance(val, Parameter) gate that selects only trainable Parameters

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Module:
    pass

def trainable_params(module):
    for name, val in module.__dict__.items():
        keep = isinstance(val, Parameter)
        if keep:
            yield name, val

m = Module()
m.w = Parameter([1.0])
m.buf = MiniTensor([0.0])
print([n for n, _ in trainable_params(m)])


def _test():
    m = Module()
    m.weight = Parameter([1.0, 2.0])
    m.bias = Parameter([0.0])
    m.running_mean = MiniTensor([0.0])  # buffer -> skip
    m.depth = 4                          # non-tensor -> skip
    names = {n for n, _ in trainable_params(m)}
    assert names == {'weight', 'bias'}
    # all yielded values are Parameters
    for _, v in trainable_params(m):
        assert isinstance(v, Parameter)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Module:
    pass

def trainable_params(module):
    for name, val in module.__dict__.items():
        keep = isinstance(val, Parameter)
        if keep:
            yield name, val

m = Module()
m.w = Parameter([1.0])
m.buf = MiniTensor([0.0])
print([n for n, _ in trainable_params(m)])
```
</details>